# Bronze Ingestion: E-Commerce Transactions (Streaming)

This notebook:
1. **Generates sample CSV data** — e-commerce transaction files written to a local staging path
2. **Streams from ADLS Gen2** — reads CSV files from a storage account using Auto Loader (cloudFiles)
3. **Writes to a bronze Delta table** — `bronze.ecommerce.transactions` with an explicit checkpoint location

### Prerequisites
- The ADLS Gen2 storage account must be accessible from the Databricks workspace
- Configure the storage account name, container, and credential parameters in the **Configuration** cell below
- The `bronze` catalog and `ecommerce` schema must exist (or be created in the setup cell)

## Configuration

In [0]:
%sql
SHOW EXTERNAL LOCATIONS

name,url,comment
adb_ai,abfss://unity-catalog-storage@dbstoragedyeqybyx3jqyi.dfs.core.windows.net/1658366818772244,null
adb_dev,abfss://unity-catalog-storage@dbstoragecvkffyaggkqri.dfs.core.windows.net/279469438045347,null
adb_train,abfss://unity-catalog-storage@dbstorage3tzw5vtlo3p7m.dfs.core.windows.net/1283281679973198,null
bronze-sa,abfss://bronze@dbstoragedyeqybyx3jqyi.dfs.core.windows.net/,null
databricks_train,abfss://unity-catalog-storage@dbstoragebtslppzvwity6.dfs.core.windows.net/3214080724152465,null
healthsaadlsrheus_outbound,abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/,Access to healthdata container on healthsaadlsrheus
test,abfss://unity-catalog-storage@dbstorageswideqgakkcy6.dfs.core.windows.net/2953091036516858,null


In [0]:
# ADLS Gen2 configuration - use healthdata container in healthsaadlsrheus storage account
STORAGE_ACCOUNT = 'healthsaadlsrheus'
CONTAINER       = 'healthdata'
SOURCE_PATH     = 'ecommerce/transactions'

# Full ADLS Gen2 source path (Auto Loader reads from here)
ADLS_SOURCE = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{SOURCE_PATH}'

# Checkpoint location (decoupled from workspace — stored in ADLS)
CHECKPOINT_PATH = 'checkpoints/bronze_ecommerce_transactions'
CHECKPOINT_LOCATION = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CHECKPOINT_PATH}'

# Target catalog and table
CATALOG = 'bronze'
SCHEMA  = 'ecommerce'
TABLE   = 'transactions'

print(f'Source:     {ADLS_SOURCE}')
print(f'Checkpoint: {CHECKPOINT_LOCATION}')
print(f'Target:     {CATALOG}.{SCHEMA}.{TABLE}')

Source:     abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions
Checkpoint: abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions
Target:     bronze.ecommerce.transactions


## Storage Authentication

Configure OAuth credentials for the ADLS Gen2 storage account.  
Credentials are retrieved from a Databricks secret scope — update the scope and key names to match your setup.

In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='adf-pipeline'),
 SecretScope(name='akv-test'),
 SecretScope(name='databricks-kv-rh-scope'),
 SecretScope(name='kvfabricprodeus2rh')]

In [0]:
# Verify access via the external location
try:
    files = dbutils.fs.ls(ADLS_SOURCE)
    print(f'Access verified via external location')
    print(f"{ADLS_SOURCE}")
except Exception as e:
    print(f'Cannot access {ADLS_SOURCE}')
    print(f'Error: {e}')
    print('External location must be properly setup before continuing')

Access verified via external location
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions


## Define the Streaming DataFrame Schema

In [0]:
from pyspark.sql.types import (
  StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# Explicit schema prevents schema inference issues across CSV batches
transactions_schema = StructType([
  StructField('order_id', StringType(), False),
  StructField('product_id', StringType(), False),
  StructField('product_name', StringType(), False),
  StructField('category', StringType(), False),
  StructField('unit_price', DoubleType(), False),
  StructField('quantity', IntegerType(), False),
  StructField('total_amount', DoubleType(), False),
  StructField('customer_id', StringType(), False),
  StructField('region', StringType(), False),
  StructField('order_timestamp', TimestampType(), False),
])

## Start the Streaming Ingestion

Uses **Auto Loader** (`cloudFiles`) to incrementally process new CSV files landing in the ADLS path.  
The checkpoint is stored in ADLS Gen2 — fully decoupled from the Databricks workspace.

In [0]:
print(f'Checkpoint location: {CHECKPOINT_LOCATION}')
print(f'ADLS Source: {ADLS_SOURCE}')

Checkpoint location: abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions
ADLS Source: abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/ecommerce/transactions


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

df_stream = (
  spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('header', 'true')
    .option('cloudFiles.schemaLocation', f'{CHECKPOINT_LOCATION}/schema')
    .schema(transactions_schema)
    .load(ADLS_SOURCE)
)

# Add ingestion metadata columns
df_enriched = (
  df_stream
    .withColumn('_ingested_at', current_timestamp())
    .withColumn('_source_file', col('_metadata.file_path'))
)

print('Streaming DataFrame schema:')
df_enriched.printSchema()

Streaming DataFrame schema:
root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



In [0]:
display(df_stream)

order_id,product_id,product_name,category,unit_price,quantity,total_amount,customer_id,region,order_timestamp
47257672-594d-4985-a222-976cc9b41268,PROD-015,Whiteboard (Magnetic),Office,79.99,5,399.95,CUST-0026,West,2026-03-09T14:38:49.897Z
98a938b1-8d8b-4cfa-a2cf-eeda2c439cd2,PROD-013,Ergonomic Mouse,Electronics,49.99,1,49.99,CUST-0044,East,2026-03-09T14:01:41.897Z
2eca3b8b-67b1-4ca4-a348-213312b6f1fb,PROD-013,Ergonomic Mouse,Electronics,49.99,4,199.96,CUST-0043,South,2026-03-09T13:54:47.897Z
ec5ee852-dca0-4551-9dd9-f14c3c82824c,PROD-020,Cable Organizer Kit,Accessories,22.99,1,22.99,CUST-0035,South,2026-03-09T14:35:33.897Z
021038b0-bf7a-4fc8-a6c6-4f61985a6c72,PROD-015,Whiteboard (Magnetic),Office,79.99,1,79.99,CUST-0043,North,2026-03-09T14:27:35.897Z
fe7b1d0b-92e4-4985-b55e-9f536538d82d,PROD-015,Whiteboard (Magnetic),Office,79.99,4,319.96,CUST-0049,Central,2026-03-09T14:35:17.897Z
737c8c73-73bd-4675-9697-67258ca08e59,PROD-019,Laptop Sleeve 15-inch,Accessories,34.99,2,69.98,CUST-0002,Central,2026-03-09T14:10:55.897Z
e6fd81dc-1c48-4e35-b631-2bf8c02ec0a4,PROD-017,USB Desk Microphone,Electronics,99.99,2,199.98,CUST-0036,East,2026-03-09T14:02:29.897Z
3cd423aa-1100-47cb-a164-f5a3b9bcd870,PROD-019,Laptop Sleeve 15-inch,Accessories,34.99,5,174.95,CUST-0002,South,2026-03-09T14:16:27.897Z
e0730094-a866-40fe-8193-76bf175a48cc,PROD-020,Cable Organizer Kit,Accessories,22.99,4,91.96,CUST-0012,East,2026-03-09T14:25:37.897Z


In [0]:
query = (
  df_enriched.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .option('mergeSchema', 'true')
    .trigger(availableNow=True)  # process all available files then stop
    .toTable(f'{CATALOG}.{SCHEMA}.{TABLE}')
)

query.awaitTermination()
print(f'Stream completed. Data written to {CATALOG}.{SCHEMA}.{TABLE}')

Stream completed. Data written to bronze.ecommerce.transactions


## Verify the Data

In [0]:
df_verify = spark.read.table(f'{CATALOG}.{SCHEMA}.{TABLE}')

print(f'Total rows: {df_verify.count()}')
print(f'\nSample records:')
df_verify.orderBy('order_timestamp', ascending=False).show(10, truncate=False)

Total rows: 600

Sample records:
+------------------------------------+----------+------------------------+-----------+----------+--------+------------+-----------+-------+-----------------------+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|order_id                            |product_id|product_name            |category   |unit_price|quantity|total_amount|customer_id|region |order_timestamp        |_ingested_at           |_source_file                                                                                                                                                                                 |
+------------------------------------+----------+------------------------+-----------+----------+--------+------------+-----------+-------+-----------------------+-----------------------+------------------------

In [0]:
from pyspark.sql.functions import count, sum as spark_sum, avg, min as spark_min, max as spark_max

print('Summary by category:')
(
  df_verify
    .groupBy('category')
    .agg(
      count('order_id').alias('order_count'),
      spark_sum('total_amount').alias('total_revenue'),
      avg('total_amount').alias('avg_order_value'),
    )
    .orderBy('total_revenue', ascending=False)
    .show(truncate=False)
)

print('Summary by region:')
(
  df_verify
    .groupBy('region')
    .agg(
      count('order_id').alias('order_count'),
      spark_sum('total_amount').alias('total_revenue'),
    )
    .orderBy('total_revenue', ascending=False)
    .show(truncate=False)
)

Summary by category:
+-----------+-----------+------------------+------------------+
|category   |order_count|total_revenue     |avg_order_value   |
+-----------+-----------+------------------+------------------+
|Electronics|322        |68110.43000000001 |211.5230745341615 |
|Office     |121        |24666.27999999999 |203.8535537190082 |
|Accessories|109        |11329.759999999997|103.94275229357795|
|Stationery |48         |1876.7400000000002|39.09875          |
+-----------+-----------+------------------+------------------+

Summary by region:
+-------+-----------+------------------+
|region |order_count|total_revenue     |
+-------+-----------+------------------+
|North  |133        |23607.109999999982|
|Central|122        |22090.36999999999 |
|South  |111        |20266.519999999993|
|West   |116        |20150.679999999993|
|East   |118        |19868.52999999999 |
+-------+-----------+------------------+



## Verify Checkpoint Location

Confirm the checkpoint files are stored in ADLS (not in the workspace temp storage).

In [0]:
print(f'Checkpoint location: {CHECKPOINT_LOCATION}')
print('\nCheckpoint contents:')
display(dbutils.fs.ls(CHECKPOINT_LOCATION))

Checkpoint location: abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions

Checkpoint contents:


path,name,size,modificationTime
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/__tmp_path_dir/,__tmp_path_dir/,0,1773064066000
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/commits/,commits/,0,1773064067000
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/metadata,metadata,45,1773064066000
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/offsets/,offsets/,0,1773064067000
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/schema/,schema/,0,1773064069000
abfss://healthdata@healthsaadlsrheus.dfs.core.windows.net/checkpoints/bronze_ecommerce_transactions/sources/,sources/,0,1773064067000
